# River Profile Chart Exploration

Given an extract from RME, extract longest level path and plot multiple indicators against distance from origin (headwater or mouth).

In [ ]:
import pandas as pd

from util.athena import query_to_dataframe


In [ ]:
def getdata() -> pd.DataFrame:
    query = """
SELECT * FROM rpt_rme_pq WHERE level_path = '55000500182580' and rme_project_id =  'cf96b00b-e452-474a-a27f-8540e990fe90'
    """
    df = query_to_dataframe(query, 'getdata')
    return df

df = getdata()

In [ ]:
df.groupby('rme_project_name').size()


In [ ]:
# print(df.head())
print(df.columns)

Index(['level_path', 'seg_distance', 'centerline_length', 'segment_area',
       'fcode', 'fcode_desc', 'longitude', 'latitude', 'ownership',
       'ownership_desc', 'state', 'county', 'drainage_area', 'stream_name',
       'stream_order', 'stream_length', 'huc12', 'rel_flow_length',
       'channel_area', 'integrated_width', 'low_lying_ratio', 'elevated_ratio',
       'floodplain_ratio', 'acres_vb_per_mile', 'hect_vb_per_km',
       'channel_width', 'lf_agriculture_prop', 'lf_agriculture',
       'lf_developed_prop', 'lf_developed', 'lf_riparian_prop', 'lf_riparian',
       'ex_riparian', 'hist_riparian', 'prop_riparian', 'hist_prop_riparian',
       'develop', 'road_len', 'road_dens', 'rail_len', 'rail_dens',
       'land_use_intens', 'road_dist', 'rail_dist', 'div_dist', 'canal_dist',
       'infra_dist', 'fldpln_access', 'access_fldpln_extent',
       'confinement_ratio', 'brat_capacity', 'brat_hist_capacity',
       'riparian_veg_departure', 'riparian_condition', 'qlow', 'q2', 's

In [ ]:
# export the data to a single parquet file
export_path = r"G:\My Drive\riverscapes_team_data_share\rpt_rme_pq-huc1706010101-levelpath55000500182580.parquet"
df.to_parquet(export_path, compression='snappy')
# saving to Google Drive is an easy way to selectively share and use in a Colab notebook
# e.g. https://colab.research.google.com/drive/1CrGNQZtojpbtGZFWk2bndf70-QuTUBDP?usp=sharing 

In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import clear_output, display
from plotly.subplots import make_subplots

# 1. FORCE Plotly to only render when we tell it to
pio.renderers.default = "notebook"

df_plot = df.sort_values('seg_distance')
numeric_cols = df_plot.select_dtypes(include='number').columns.tolist()
if 'seg_distance' in numeric_cols:
    numeric_cols.remove('seg_distance')

default_y1 = 'channel_width' if 'channel_width' in numeric_cols else numeric_cols[0]
default_y2 = 'drainage_area' if 'drainage_area' in numeric_cols else (numeric_cols[1] if len(numeric_cols) > 1 else numeric_cols[0])

y1_dropdown = widgets.Dropdown(options=numeric_cols, value=default_y1, description='Primary Y:')
y2_dropdown = widgets.Dropdown(options=numeric_cols, value=default_y2, description='Secondary Y:')

out = widgets.Output()

def render_plot(y1_col, y2_col):
    with out:
        clear_output(wait=True)
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        
        fig.add_trace(
            go.Scatter(x=df_plot['seg_distance'], y=df_plot[y1_col], name=y1_col, mode="lines"),
            secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(x=df_plot['seg_distance'], y=df_plot[y2_col], name=y2_col, mode="lines"),
            secondary_y=True,
        )
        
        fig.update_layout(title_text=f"{y1_col} and {y2_col} along Segment Distance")
        # 2. Use display(fig) which Plotly now handles correctly thanks to the renderer change
        display(fig)

def on_dropdown_change(change):
    # Only trigger if the value actually changed
    if change['type'] == 'change' and change['name'] == 'value':
        render_plot(y1_dropdown.value, y2_dropdown.value)

y1_dropdown.observe(on_dropdown_change, names='value')
y2_dropdown.observe(on_dropdown_change, names='value')

# 3. Order matters: Display the container first, THEN trigger the first plot
ui = widgets.HBox([y1_dropdown, y2_dropdown])
display(ui)
display(out)

render_plot(default_y1, default_y2)